# Predictia performantei ofensive a jucatorilor

Acest notebook pregateste datele jucator–meci si compara regresia logistica cu padurea aleatoare. Tinta este **cel putin un gol sau assist in meci**.

Pentru a evita scurgerea de informatii, modelul foloseste doar date din **meciurile anterioare** ale jucatorului. Minutele, golurile, assisturile, suturile si cartonasele devin caracteristici calculate pe ultimele cinci aparitii; golurile si assisturile din meciul curent definesc doar tinta.


## 1. Date de intrare

Ruleaza mai intai `data_scraper.ipynb`. Alege `MODEL_SOURCE` in celula de configurare pentru a folosi exportul FBref sau StatsBomb. Ambele fisiere au aceleasi coloane pentru jucator, data, minute, goluri, assisturi, suturi si cartonase. Datele meciului curent definesc doar tinta; caracteristicile provin din aparitiile anterioare.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
if CWD.name == "notebooks" and (CWD.parent / "README.md").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "notebooks").is_dir() and (CWD / "README.md").exists():
    PROJECT_ROOT = CWD
else:
    candidates = [p for p in (CWD, *CWD.parents) if (p / "notebooks").is_dir() and (p / "README.md").exists()]
    if not candidates:
        raise RuntimeError("Deschide Jupyter din IAPD_project/ sau din directorul notebooks/.")
    PROJECT_ROOT = candidates[0]

DATA_ROOT = PROJECT_ROOT / "notebooks" / "data"
MODEL_SOURCE = "statsbomb"  # Alege "fbref" pentru exportul FBref.
INPUT_FILES = {
    "fbref": DATA_ROOT / "raw" / "fbref" / "fbref_player_match_stats.csv",
    "statsbomb": DATA_ROOT / "raw" / "statsbomb" / "statsbomb_player_match_stats.csv",
}
if MODEL_SOURCE not in INPUT_FILES:
    raise ValueError(f"Unknown MODEL_SOURCE: {MODEL_SOURCE}")
INPUT_FILE = INPUT_FILES[MODEL_SOURCE]
OUTPUT_DIR = DATA_ROOT / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Asociaza fiecare camp cu numele coloanei din setul de date ales.
# Ambele exporturi data_scraper contin aceste campuri comune.
COLUMN_MAP = {
    "player": "player",
    "match_date": "date",
    "minutes": "Min",
    "goals": "Gls",
    "assists": "Ast",
    "shots": "Sh",
    "yellow_cards": "CrdY",
    "red_cards": "CrdR",
    # Optional: "match_id": "game_id",
}

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Fisierul de intrare nu a fost gasit: {INPUT_FILE}. Ruleaza mai intai notebookul data_scraper, "
        "sau actualizeaza INPUT_FILE catre fisierul CSV jucator-meci."
    )

raw = pd.read_csv(INPUT_FILE)
MIN_DATA_ROWS = 1_000
if len(raw) < MIN_DATA_ROWS:
    raise ValueError(f"Criteriul setului de date nu este indeplinit: {len(raw):,} randuri de intrare; sunt necesare cel putin {MIN_DATA_ROWS:,}.")
print(f"Incarcate {len(raw):,} randuri si {len(raw.columns):,} coloane")
print("Available coloane:", list(raw.columns))
raw.head()

## 2. Standardizarea coloanelor

Maparea de mai jos este explicita, deoarece sursele pot avea antete diferite. Daca lipseste un camp obligatoriu, actualizeaza `COLUMN_MAP` in celula de configurare. Valorile lipsa la cartonase raman lipsa, pentru a nu le interpreta automat ca zero.


In [ ]:
required = ["player", "match_date", "minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
missing_mapping = [key for key in required if COLUMN_MAP.get(key) not in raw.columns]
if missing_mapping:
    raise KeyError(
        "These mapped coloane were not found: " + ", ".join(f"{key} -> {COLUMN_MAP.get(key)!r}" for key in missing_mapping)
        + ". Actualizeaza COLUMN_MAP cu antetele exacte de mai sus."
    )

rename = {source: canonical for canonical, source in COLUMN_MAP.items() if source in raw.columns}
data = raw.rename(columns=rename).copy()
if "match_id" in COLUMN_MAP and COLUMN_MAP["match_id"] in raw.columns:
    data = data.rename(columns={COLUMN_MAP["match_id"]: "match_id"})

data["match_date"] = pd.to_datetime(data["match_date"], errors="coerce", utc=True)
numeric = ["minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
for col in numeric:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data["player"] = data["player"].astype("string").str.strip()
data = data.dropna(subset=["player", "match_date"]).copy()
data = data[data["minutes"].fillna(0) > 0].copy()
data = data.sort_values(["player", "match_date"]).reset_index(drop=True)

print(f"Randuri jucator-meci utilizabile: {len(data):,}")
print("Valori lipsa pe camp:")
display(data[required].isna().sum().to_frame("missing_rows"))

## 3. Definirea tintei si crearea caracteristicilor pre-meci

Performanta ofensiva ridicata inseamna **cel putin un gol sau assist in meciul curent**. Caracteristicile sunt calculate din ultimele cinci aparitii anterioare, cu decalare de un meci, astfel incat statisticile curente sa nu intre in predictie. Jucatorul trebuie sa aiba cel putin o aparitie anterioara.


In [ ]:
data = data.dropna(subset=["goals", "assists"]).copy()
data["goal_contributions"] = data["goals"] + data["assists"]
data["high_offensive_performance"] = (data["goal_contributions"] >= 1).astype(int)

past_fields = ["minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
for field in past_fields:
    data[f"recent5_mean_{field}"] = (
        data.groupby("player", sort=False)[field]
        .transform(lambda series: series.shift(1).rolling(window=5, min_periods=1).mean())
    )
    data[f"recent5_sum_{field}"] = (
        data.groupby("player", sort=False)[field]
        .transform(lambda series: series.shift(1).rolling(window=5, min_periods=1).sum())
    )

feature_cols = [col for col in data.columns if col.startswith("recent5_")]
model_data = data.dropna(subset=feature_cols + ["high_offensive_performance"]).copy()
print("Distributia tintei (0 = fara contributie la gol, 1 = cel putin una):")
display(model_data["high_offensive_performance"].value_counts(normalize=True).rename("share").to_frame())
if len(model_data) < MIN_DATA_ROWS:
    raise ValueError(
        f"Criteriul pentru setul de modelare nu este indeplinit: {len(model_data):,} randuri jucator-meci utilizabile; "
        f"need at least {MIN_DATA_ROWS:,}."
    )
if len(feature_cols) < 10:
    raise ValueError(f"Criteriul caracteristicilor modelului nu este indeplinit: {len(feature_cols)} engineered features; sunt necesare cel putin 10.")
print(f"OK — setul de modelare are {len(model_data):,} usable randuri si {len(feature_cols):,} caracteristici pre-meci")

## 4. Impartirea cronologica antrenare/test

Setul de test contine cele mai recente 20% dintre datele meciurilor. Aceasta impartire aproximeaza predictia unor meciuri viitoare mai bine decat o impartire aleatoare. Ambele clase trebuie sa existe atat in setul de antrenare, cat si in cel de test.


In [ ]:
model_data = model_data.sort_values("match_date").reset_index(drop=True)
unique_dates = model_data["match_date"].drop_duplicates().sort_values().reset_index(drop=True)
split_date_index = int(len(unique_dates) * 0.8)
if split_date_index <= 0 or split_date_index >= len(unique_dates):
    raise ValueError("Nu sunt suficiente date distincte pentru impartirea cronologica 80/20.")
split_date = unique_dates.iloc[split_date_index]

# Keep all matches on the same date together to avoid splitting one match day across sets.
train = model_data[model_data["match_date"] < split_date].copy()
test = model_data[model_data["match_date"] >= split_date].copy()
X_train, X_test = train[feature_cols], test[feature_cols]
y_train = train["high_offensive_performance"]
y_test = test["high_offensive_performance"]

if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError(
        "Impartirea cronologica nu contine ambele clase ale tintei in antrenare si test. "
        "Foloseste mai multe sezoane/meciuri sau modifica definitia tintei."
    )
print(f"Randuri de antrenare: {len(train):,} through {train['match_date'].max().date()}")
print(f"Randuri de test: {len(test):,} from {test['match_date'].min().date()}")

## 5. Antrenarea si compararea modelelor

Ambele modele completeaza valorile lipsa din caracteristici cu mediana. Regresia logistica isi si scaleaza intrarile. Ponderarea claselor ajuta cand contributiile la goluri sunt rare. Comparatia raporteaza precizia, rata de identificare, scorul F1, ROC AUC si media preciziei (PR AUC).


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

models = {
    "Regresie logistica": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ]),
    "Padure aleatoare": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, class_weight="balanced",
            random_state=42, n_jobs=-1,
        )),
    ]),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    predictions[name] = (predicted, probability)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted, zero_division=0),
        "f1": f1_score(y_test, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision": average_precision_score(y_test, probability),
    })

comparison = pd.DataFrame(results).set_index("model").sort_values("average_precision", ascending=False)
display(comparison.round(3))

## 6. Evaluarea celui mai bun model si salvarea rezultatelor

Modelele sunt ordonate dupa media preciziei, utila cand clasa pozitiva este rara. Verifica matricea de confuzie, precizia si rata de identificare inainte de a interpreta rezultatul. Scorurile depind de campionat, sezoane si acoperirea sursei alese.


In [ ]:
best_name = comparison.index[0]
best_model = models[best_name]
best_pred, best_probability = predictions[best_name]
print("Selectat dupa media preciziei:", best_name)
print("Confusion matrix (rows = actual [0, 1], coloane = predicted [0, 1]):")
print(confusion_matrix(y_test, best_pred))

comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", encoding="utf-8-sig")
test_results = test[["player", "match_date", "goals", "assists", "high_offensive_performance"]].copy()
test_results["predicted_high_performance"] = best_pred
test_results["predicted_probability"] = best_probability
test_results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
print("Salvat:", OUTPUT_DIR / "model_comparison.csv")
print("Salvat:", OUTPUT_DIR / "test_predictions.csv")

## Interpretare si limitari

- Contributia la gol intr-un meci este o tinta simpla si reproductibila, dar nu este o definitie universala a performantei ofensive.
- Modelul foloseste doar forma recenta. Nu include primul unsprezece estimat, forta adversarului, accidentarile sau minutele estimate inainte de meci.
- Acoperirea si definitiile statisticilor difera intre furnizori. Nu combina sursele inainte de a reconcilia identitatea jucatorilor, meciurile, datele si definitiile statisticilor.
- Rezultatele nu sunt direct comparabile intre campionate sau sezoane fara date reprezentative. Raporteaza sursa, intervalul de date, volumul, proportia claselor si metricile.
